In [1]:
import pandas as pd


In [2]:
df = pd.read_csv("IMDB Dataset.csv")



In [3]:
df.head()



,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
df.isnull().sum()



review       0
sentiment    0
dtype: int64

In [5]:

df.drop_duplicates(inplace=True)

In [6]:
df.shape



(49582, 2)

### Preprocessing

### 1. Convertngg to loercase

In [7]:
df["review"] = df["review"].str.lower()





### 2. removing the URLs


In [8]:
import re


def remove_urls(text):
    text = re.sub(r"http\S+" , "", text)
    return text
df["review"] = df["review"].apply(remove_urls)   

### 3. remove puntuation s

In [9]:
def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]" , "" , text)
    return text


df["review"] = df["review"].apply(remove_punctuations)
    

### 4 Remove HTML tags


In [11]:
def remove_html(text):
    text = re.sub(r"<.*>" ,"" , text)
    return text


df["review"] = df["review"].apply(remove_html)

### 5. removing stopwords

In [12]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\priya\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\priya\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\priya\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [13]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [14]:
def remove_stopwords(text):
    tokens = word_tokenize(text.lower())

    
    stop_words = set(stopwords.words("english"))

    filtered_words = []

    for word in tokens:
        if word not in stop_words:
            filtered_words.append(word)

    return " ".join(filtered_words)

df["review"] = df["review"].apply(remove_stopwords)


In [12]:
df.head()

,review,sentiment
0,one reviewers mentioned watching 1 oz episode ...,positive
1,wonderful little production br br filming tech...,positive
2,thought wonderful way spend time hot summer we...,positive
3,basically theres family little boy jake thinks...,negative
4,petter matteis love time money visually stunni...,positive


### 6. Stemming

In [12]:
# running --> run
from nltk.stem import PorterStemmer

In [13]:
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)

    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

In [16]:
### 7.Encoding

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])


In [17]:
y = df["sentiment"]

In [20]:
# 8. Vectorization
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)

X = tf.fit_transform(df["review"])



### Dataset and DataLoaders

In [25]:
from sklearn.model_selection import train_test_split

X_train, X_test , y_train , y_test = train_test_split(
    X, y , test_size =0.2,  random_state= 42)
    

In [26]:
X_train.shape

(39665, 5000)

In [31]:
import torch

from torch.utils.data import TensorDataset, DataLoader


In [33]:
# No conversion needed
print(type(X_train))
print(type(X_test))

<class 'numpy.ndarray'>
<class 'numpy.ndarray'>


In [35]:
from torch.utils.data import TensorDataset
import torch

train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [36]:
train_loader = DataLoader(train_set , shuffle = True , batch_size = 64)
test_loader = DataLoader(test_set , shuffle= True , batch_size= 64)


### Builld RNN

In [37]:
import torch.nn as nn
import torch.optim as optim


In [ ]:

import torch
import torch.nn as nn

class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.rnn = nn.RNN(
            input_size,
            hidden_size,
            num_layers,
            batch_first=True
        )

        # Fully connected layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        h0 = torch.zeros(
            self.num_layers,
            x.size(0),
            self.hidden_size,
            device=x.device
        )

        out, _ = self.rnn(x, h0)

        out = self.fc(out[:, -1, :])

        return out



In [ ]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion  = nn.BCELoss()

optimizer = optim.Adam(model.parameters()

###Trainingg the RNN

In [ ]:
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)

        outputs = torch.sigmoid(outputs.squeeze())

        loss = criterion(outputs, yb.float())

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(f"Epoch {epoch + 1}/{num_epochs}, Loss = {avg_loss:.4f}")
    


        

In [ ]:
model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0

    for Xb, yb in test_loader:
        Xb = Xb.squeeze(1)

        outputs = model(Xb)

        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    accuracy = (correct_vals / tot_vals) * 100

    print(f"Accuracy = {accuracy:.2f}%")
        
